In [1]:
import time
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL="http://books.toscrape.com/"

print("Imports Successful! pandas version:",pd.__version__)

Imports Successful! pandas version: 3.0.5


In [3]:
from requests import Response
def get_soup(url):
    """
    Fetches the HTML of a URL and returns a BeautifulSoup object.
    Includes a polite sleep timer to not overload the site.
    """
    time.sleep(0.5)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36"
    }

    try:
        response=requests.get(url,headers=headers,timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text,"html.parser")
    except requests.RequestException as e:
        print(f"Error fetching{url}: {e}")
        return None


In [ ]:
def get_categories():
    """
    Scrapes the homepage sidebar and returns a dictionary of:
    { "Category Name": "Full Category URL" }
    """
    soup=get_soup(BASE_URL)
    if not soup:
        return{}
    
    categories={}
    #CSS selectors trageting all categories links in the sidebar 
    category_links=soup.select(".side_categories ul li ul li a")

    for tag in category_links:
        category_name=tag.text.strip()
        # Convert relative link to a full URL
        category_url=urljoin(BASE_URL,tag["href"])
        categories[category_name]=category_url

    return categories
# Sample test function and dispaly the categories names 5
all_categories =get_categories()
print(f"Total categories found:{len(all_categories)}\n")    

for name,url in list(all_categories.items())[:5]:
    print(f"!{name} -> {url}")






Total categories found:50

!Travel -> http://books.toscrape.com/catalogue/category/books/travel_2/index.html
!Mystery -> http://books.toscrape.com/catalogue/category/books/mystery_3/index.html
!Historical Fiction -> http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
!Sequential Art -> http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
!Classics -> http://books.toscrape.com/catalogue/category/books/classics_6/index.html


In [7]:
def partse_book_item(pod,category_name):
    """
    Extracts raw data fields from a single <article class="product_pod"> element.
    """
     # 1. Title: The <a> tag inside <h3> has a 'title' attribute with the full, untruncated name
    title_tag=pod.select_one("h3 a")
    title=title_tag["title"].strip() if (title_tag and title_tag.has_attr('title')) else (title_tag.text.strip() if title_tag else None)

    #2. raw price string 
    price_tag=pod.select_one(".price_color")
    price=price_tag.text.strip() if price_tag else None 

    #3 star rateing
    rating_tag=pod.select_one("p.star-rating")
    rating=None
    if rating_tag and rating_tag.has_attr("class"):
         # rating_tag["class"] gives a list like: ['star-rating', 'Three']
        rating_classes = [c for c in rating_tag["class"] if c != "star-rating"]
        if rating_classes:
            rating = rating_classes[0]

    #4 Availability 
    avail_tag=pod.select_one(".availability")
    availability=avail_tag.text.strip() if avail_tag else None 

    return {
        "title": title,
        "price": price,
        "star_rating": rating,
        "availability": availability,
        "category": category_name
    }     
 
